## Loading the data
Load raw posts as .csv from Azure blob storage.

In [ ]:
from dotenv import load_dotenv
import os
load_dotenv()
import pandas as pd
from openai import OpenAI

openai_api_key = os.getenv("OPENAI_KEY") # Your API key here

gold_label = 'topic'

dataset_name = "bills-hoyle-2022" # "bills-hoyle-2022" or "wikitext-103-hoyle-2022"
test_data = pd.read_json(f"../data/{dataset_name}/test.metadata.jsonl", lines=True)

In [ ]:
test_data.head()

In [ ]:
%matplotlib inline

In [ ]:
import logging
for handler in logging.root.handlers[:]:
   logging.root.removeHandler(handler)
logging.basicConfig(format='%(asctime)s : %(levelname)s : %(message)s',
                   level=logging.NOTSET,
                   filename='../data/logs/lda-logs.log')

## Apply pre-processing
Note, first the contents are still full text and the mean length is in chars, after preprocessing it into bag of words it is a list of strings and length is in word count therefor

In [ ]:
input_column = "tokenized_text"

test_docs = test_data[input_column].tolist()
test_docs = [doc.split() for doc in test_docs]
print(len(test_docs))
print(test_docs[0][:200])

In [ ]:
# Remove rare and common tokens.
from gensim.corpora import Dictionary

# Create a dictionary representation of the documents.
dictionary = Dictionary(test_docs)

# Bag-of-words representation of the documents.
test_corpus = [dictionary.doc2bow(doc) for doc in test_docs]

print('Number of unique tokens: %d' % len(dictionary))
print('Number of documents: %d' % len(test_corpus))

# Training the model: LDA

In [ ]:
run = "paper-bm-lda-v1-bills"

In [ ]:
import mlflow

mlflow.set_tracking_uri("http://localhost:8080")

In [ ]:
def get_or_create_experiment(experiment_name):
    """
    Retrieve the ID of an existing MLflow experiment or create a new one if it doesn't exist.

    This function checks if an experiment with the given name exists within MLflow.
    If it does, the function returns its ID. If not, it creates a new experiment
    with the provided name and returns its ID.

    Parameters:
    - experiment_name (str): Name of the MLflow experiment.

    Returns:
    - str: ID of the existing or newly created MLflow experiment.
    """

    if experiment := mlflow.get_experiment_by_name(experiment_name):
        return experiment.experiment_id
    else:
        return mlflow.create_experiment(experiment_name)

In [ ]:
llmbt_experiment_id = get_or_create_experiment("ICNLSP Experiment LDA")
llmbt_experiment_id

In [ ]:
import numpy as np

def get_document_vectors(lda_model, corpus, num_topics):
    doc_vectors = []
    
    for bow in corpus:
        topic_distribution = lda_model.get_document_topics(bow, minimum_probability=0)
        # Convert sparse topic distribution into a dense vector
        dense_vector = np.zeros(num_topics)
        for topic_id, prob in topic_distribution:
            dense_vector[topic_id] = prob
        doc_vectors.append(dense_vector)
    
    return np.array(doc_vectors)

In [ ]:
from typing import List
import json

from openai import OpenAI
from gensim.models.ldamulticore import LdaModel
from wordcloud import WordCloud
from textonomy.utils import metrics
from sklearn.feature_extraction.text import CountVectorizer
from sentence_transformers import SentenceTransformer
import os
import mlflow
from mlflow.models import infer_signature
import mlflow.data
os.environ["TOKENIZERS_PARALLELISM"] = "true"

mlflow.openai.autolog()

openai = OpenAI(api_key=openai_api_key)  # Make sure openai_api_key is defined

# Run the LLM+BERTopic model

# Embeddings for internal clustering metric
embedding_model = SentenceTransformer('all-mpnet-base-v2') # or other modernbert model
embeddings = embedding_model.encode(test_docs)

ks = [25,23,22]

for i, k in enumerate(ks):

    model_name = "lda"
    printprefix = str(i) + " - " + model_name + ": "
    training_info_tag = "bills attempt #1"

    with mlflow.start_run(experiment_id=llmbt_experiment_id):
        # Set a tag that we can use to remind ourselves what this run was for
        mlflow.set_tag(
            "Training Info", training_info_tag
        )
        mlflow.log_param("model_name", model_name)

        dataset = mlflow.data.from_pandas(test_data, source="Azure Blob Storage", name=dataset_name)
        mlflow.log_input(dataset, context=dataset_name)

        topics_df, topics, topic_probs = None, None, None

        model_params = {
        "k": k,
        "language": "english",
        "chunksize": 2000,
        "passes": 20,
        "iterations": 50,
        "eval_every": 1,
        }

        mlflow.log_params(model_params)

        print(printprefix + "LDA model fitting")

        # Make an index to word dictionary.
        temp = dictionary[0]
        id2word = dictionary.id2token

        model = LdaModel(
            corpus=test_corpus,
            id2word=id2word,
            chunksize=model_params["chunksize"],
            alpha='auto',
            eta='auto',
            iterations=model_params["iterations"],
            num_topics=model_params["k"],
            passes=model_params["passes"],
            eval_every=model_params["eval_every"],
        )

        print(printprefix + "LDA model fitting done")
        print(printprefix + "Calculating topics for test data")

        test_data['predicted_label'] = [max(model.get_document_topics(doc), key=lambda x: x[1])[0] for doc in test_corpus]
        
        # Log clustered test data to mlflow
        clustered_test_data = "../data/tmp/clustered_test_data.json"
        test_data[['id', gold_label, 'predicted_label']].to_json(clustered_test_data, orient="records")
        mlflow.log_artifact(local_path=clustered_test_data, artifact_path="clustered_test_data.json")
        os.remove(clustered_test_data)

        topics = test_data['predicted_label'].to_list()
        #topic_probs = get_document_vectors(model, test_corpus, model_params["k"])

        print(printprefix + "Calculating metrics")

        purpose = "A taxonomy to organise these summaries of US congressional bills into (main) topics. Aim at around 15-25 topics. Examples: 'Government Operations', 'Environment'..."
        topk = 10

        metric_dict, llm_usefulness_reasoning = metrics.evaluate_lda_model(lda_model=model, purpose=purpose, openai_client=openai, topk=topk, pp_ref_docs=test_docs, model_labels=topics, gold_labels=test_data[gold_label].to_list(), embeddings=embeddings)
        
        # Log metrics
        mlflow.log_param("eval_topk", topk)

        metric_dict['outlier_ratio'] = 0
        metric_dict['num_test_docs'] = test_data.id.nunique()
        metric_dict['num_topics'] = model_params["k"]
        metric_dict['perplexity'] = model.log_perplexity(test_corpus)
        
        valid_metrics, invalid_metrics = {}, {}
        for k, v in metric_dict.items():
            try:
                valid_metrics[k] = float(v)
            except (ValueError, TypeError):
                invalid_metrics[k] = v  # Store invalid values separately
        # Log valid metrics
        mlflow.log_metrics(valid_metrics)
        # If there are invalid metrics, save them to a text file and log as an artifact
        if invalid_metrics:
            invalid_metrics_path = "../data/tmp/invalid_metrics.txt"
            with open(invalid_metrics_path, "w") as f:
                json.dump(invalid_metrics, f, indent=4)  # Save as JSON for readability
            mlflow.log_artifact(invalid_metrics_path)  # Log the file in MLflow
            print(printprefix + f"Invalid metrics saved to {invalid_metrics_path}")
            os.remove(invalid_metrics_path)  # Remove the local file
        
        mlflow.log_text(purpose, "user_defined_purpose_for_U_LLM_USEFULNESS.txt")
        mlflow.log_text(str(llm_usefulness_reasoning), "U_LLM_USEFULNESS_reasoning.txt")

        # Print metrics
        #metric_df = create_metrics_dataframe(metric_dict)
        metric_df = pd.DataFrame.from_dict(metric_dict, orient="index", columns=["value"])
        display(metric_df)

        ## Log plots and data
        # Log topic describtions
        topic_info_list = model.show_topics(num_topics=-1, num_words=10, formatted=True)
        topic_info_txt_path = "../data/tmp/topics_info.txt"
        with open(topic_info_txt_path, "w") as f:
            for item in topic_info_list:
                f.write(f"{item}\n")
        mlflow.log_artifact(topic_info_txt_path, artifact_path="lda_topic_info_data")

        # Create a word cloud for one of the topics
        topic_words = model.show_topic(0, topn=20)
        wordcloud = WordCloud(width=800, height=400).generate_from_frequencies(dict(topic_words))

        # Save the plot to a file
        wordcloud_file = "../data/tmp/wordcloud.png"
        wordcloud.to_file(wordcloud_file)

        # Log the plot as an artifact
        mlflow.log_artifact(wordcloud_file)
        
        # Infer signature based on the data and model
        signature = infer_signature(test_corpus, model.get_document_topics(test_corpus))

        # Log the model with signature
        model_info = mlflow.sklearn.log_model(model, "lda_model", signature=signature)

        print("Logged run '" + str(run) + "' to MLflow.")
        print(f"Model URI: {model_info.model_uri}")
        print(f"MLflow Tracking URI: {mlflow.get_tracking_uri()}")

In [ ]:
import os
from openai import OpenAI
from textonomy.models import Textonomy
os.environ["TOKENIZERS_PARALLELISM"] = "true"

openai_api_key = os.getenv("OPENAI_KEY") # Your API key here
client = OpenAI(api_key=openai_api_key)

model = Textonomy()

# Calculate LDA Stability Baseline

In [ ]:
import itertools
from typing import Dict

artifact_path = "clustered_test_data.json/clustered_test_data.json"

run_ids = [
    "18c174adc4e6417da79958725d738258",
    "4c74e6fcf3c44280bd9e9dc06427e4c3",
    "24be78b4967548f3bf487446d698ead0",
    "5fc86fab2380486cbbabbe3162dbc18e",
    "3d6e37eb12a34787aa03d873c57453c3",
    "536cdcaa398a43debaf57f35258279b5",
    "137919f5145c41d4b7c9197411e52832",
    "d0cfc32aaaf94483aef38703306914a0",
    "23bd60d0aa1c401780d7db6f582400cf",
    "a9fcd6319b6a4bc98800b04fd8d300ed"
]

artifact_uris = []
for run_id in run_ids:
    try:
        artifact_uri = mlflow.artifacts.download_artifacts(run_id=run_id, artifact_path=artifact_path)
        artifact_uris.append(artifact_uri)
    except Exception as e:
        print(f"Error downloading artifact for run_id {run_id}: {e}")
        # Handle the error appropriately, e.g., skip this run or raise the exception
        continue  # Skip to the next run ID


dataframes = []
for artifact_uri in artifact_uris:
    try:
        with open(artifact_uri, "r") as f:
            data = json.load(f)
        df = pd.DataFrame(data)
        dataframes.append(df)
    except Exception as e:
            print(f"Error loading or processing artifact from {artifact_uri}: {e}")
            continue

if len(dataframes) < 2:
    raise ValueError(f"Fewer than 2 useable artifacts, could not create comparison pairs")

# Initialize a dictionary to store the sum of each metric and a counter
metric_sums: Dict[str, float] = {}
metric_counts: Dict[str, int] = {}
metric_dicts = []

# Iterate over all pairs of runs
for df1, df2 in itertools.combinations(dataframes, 2):
    try:
        # Calculate external cluster metrics
        metrics = model.calculate_external_cluster_metrics(y_true=df1['predicted_label'], y_pred=df2['predicted_label'])
        metric_dicts.append(metrics)

        # Accumulate the metrics
        for metric_name, metric_value in metrics.items():
            metric_sums[metric_name] = metric_sums.get(metric_name, 0) + metric_value
            metric_counts[metric_name] = metric_counts.get(metric_name, 0) + 1
    except Exception as e:
            print(f"Error calculating or processing metrics: {e}")
            continue

# Calculate average metrics
average_metrics = {}
for metric_name in metric_sums:
    average_metrics[metric_name] = metric_sums[metric_name] / metric_counts[metric_name]

print("Average metrics:")
for metric_name, metric_value in average_metrics.items():
    print(f"{metric_name}: {metric_value}")


In [ ]:
metric_dicts

In [ ]:
from sklearn.metrics.cluster import adjusted_rand_score, normalized_mutual_info_score
import numpy as np

def purity(model_labels, gold_labels): 
    """
    Calculates the Purity metric as described in https://aclanthology.org/P16-1110/
    "ALTO: Active Learning with Topic Overviews for Speeding Label Induction and Document Labeling"
    
    For sanity check - The purity of any two user labels should be 1
    """
    assert len(model_labels) == len(gold_labels)

    # somewhat faster than a pure-python implementation
    purity_sum = (
        pd.DataFrame({"pred": model_labels, "true": gold_labels})
          .groupby(["pred", "true"], as_index=False)
          .size()
          .groupby("pred")["size"]
          .max()
          .sum()
    )
    return purity_sum / len(model_labels)

# Ground truth and predicted labels
y_true = test_data[gold_label].to_list()
y_pred = topics

# Compute metrics
ari = adjusted_rand_score(y_true, y_pred)
nmi = normalized_mutual_info_score(y_true, y_pred)
purity1 = purity(model_labels=y_pred, gold_labels=y_true)

# Print results
print(f"ARI: {ari:.4f}")
print(f"NMI: {nmi:.4f}")
print(f"Purity: {purity1:.4f}")


In [ ]:
model.show_topics(num_topics=-1, num_words=10, formatted=True)